# Lezione 09 — Distribuzione e pubblicazione

**Corso:** MLOps e Monitoraggio con Python · **Blocco B**
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/09_despliegue_publicacion_IT.md`

## Cosa vedrai in questa lezione

- I tre **ambienti** di un sistema reale (sviluppo, staging, produzione) e a cosa serve ciascuno, con un esempio concreto.
- Riaddestrare le due versioni del modello di churn (v1 senza bilanciare, v2 bilanciata) delle lezioni 4, 7 e 8.
- Costruire, sopra l'API della lezione 7, due versioni attive nello stesso momento (`/predecir_v1`, `/predecir_v2`).
- Simulare in codice reale tre strategie di distribuzione con un generatore di traffico sintetico riproducibile: **canary**, **shadow** e **A/B**.
- Perché l'A/B usa un **hash deterministico** dell'`id_cliente` e non un'estrazione casuale pura.
- Implementare e provare una funzione di **rollback** con un criterio di decisione esplicito e un log reale dell'azione.

Tutto avviene dentro una cartella di lavoro effimera (`proyecto_churn/`), creata all'inizio ed **eliminata alla fine** di questo stesso notebook. In questa lezione non si usa MLflow: non serve per simulare traffico e decidere un rollback.

## Prerequisiti

- Dalla lezione 7: l'API FastAPI (`Cliente`, `TestClient`) che questo notebook amplia a due versioni.
- Dalla lezione 8: il registro dei modelli con alias `candidato`/`producción`/`archivado` -- questa lezione simula, con traffico reale, la decisione che giustificherebbe promuovere l'una o l'altra.
- Dalle lezioni 4 e 5: la configurazione vincitrice del modello (regressione logistica, `class_weight="balanced"`, `C=1,0`) e il suo confronto con la versione senza bilanciare.
- Python di base: funzioni, dizionari, `Counter`; non serve aver usato prima `hashlib` né test statistici.

> **Come usare questo notebook:** esegui le celle in ordine, dall'alto in basso, una sola volta. Richiede meno di un minuto in totale.

---
## Installazione

Installa solo ciò che manca (FastAPI, httpx, uvicorn, scipy).

In [ ]:
import warnings

# Silenziamo un avviso di compatibilità di starlette/httpx (fisso, non aggiunge nulla qui) perché
# l'output di questo notebook non dipenda da dettagli interni di versioni delle librerie
warnings.filterwarnings("ignore", category=DeprecationWarning)

import subprocess
import sys


def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("scipy", "scipy")
asegurar("joblib", "joblib")
asegurar("fastapi", "fastapi")
asegurar("httpx", "httpx")
asegurar("uvicorn", "uvicorn")
print("Librerie pronte.")

---
## Cella 0 — Setup

In [ ]:
import os
import shutil
import hashlib
import json
from collections import Counter

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
COLUMNAS_ENTRADA = COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS

print("Librerie e funzioni ausiliarie pronte.")

---
## Cartella di lavoro effimera

In [ ]:
CARPETA = "proyecto_churn"

# Puliamo e ricreiamo la cartella perché il notebook sia ripetibile da zero (stessa convenzione delle lezioni 7 e 8)
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Cartella di lavoro creata:", os.path.abspath(CARPETA))

---
## 1. I tre ambienti: sviluppo, staging, produzione

Prima di ripartire il traffico tra versioni, conviene distinguere DOVE gira ciascuna cosa. I tre ambienti condividono lo stesso schema di dati e, in un progetto serio, lo stesso codice -- ciò che cambia è il volume di dati, chi li usa e il rischio di un errore:

| Ambiente | A cosa serve | Chi lo usa | Rischio di un errore |
|---|---|---|---|
| **Sviluppo** | Iterare rapidamente: provare un'idea, fare debug, eseguire il notebook di questa lezione | chi programma | basso -- riguarda solo chi lo esegue |
| **Staging** | Una replica di produzione con dati realistici, per provare il `Dockerfile`/`ci.yml` della lezione 8 prima di toccare traffico reale | il team, prima di pubblicare | medio -- se fallisce, non arriva a un cliente |
| **Produzione** | Serve richieste reali di clienti reali | i clienti, senza saperlo | alto -- un errore qui è un incidente vero |

Il notebook lo rende concreto: `sviluppo` usa un campione di 50 righe di `churn.csv`, `staging` usa il 20 % delle righe, e `produzione` usa l'insieme completo -- STESSE colonne nei tre, così che un `Cliente` validato in sviluppo lo resta anche in produzione.

In [ ]:
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)

# Uno stesso schema (le stesse colonne), tre quantità di dati diverse -- una differenza reale e verificabile tra ambienti
ENTORNOS = {
    "desarrollo": {
        "datos": df.sample(n=50, random_state=SEMILLA),
        "descripcion": "portatile dello sviluppatore, sottoinsieme piccolo, iterazione rapida",
    },
    "staging": {
        "datos": df.sample(frac=0.2, random_state=SEMILLA),
        "descripcion": "replica di produzione con un campione realistico di dati, senza traffico di clienti reali",
    },
    "produccion": {
        "datos": df,
        "descripcion": "serve richieste reali; qualsiasi errore qui riguarda un cliente vero",
    },
}

for nombre, info in ENTORNOS.items():
    print(f"ambiente {nombre}: {len(info['datos'])} righe -- {info['descripcion']}")

assert set(ENTORNOS["staging"]["datos"].columns) == set(ENTORNOS["produccion"]["datos"].columns), \
    "staging deve condividere lo stesso schema di produzione"
assert len(ENTORNOS["desarrollo"]["datos"]) < len(ENTORNOS["staging"]["datos"]) < len(ENTORNOS["produccion"]["datos"])
print("I tre ambienti condividono lo schema; cambiano solo il volume di dati e il rischio di un errore.")

> **Regola pratica:** se un cambiamento non è stato provato in staging con dati che somiglino a quelli reali, non è pronto per la produzione -- per quanto sia andato veloce in sviluppo. Staging esiste proprio per intercettare il "sul mio portatile funzionava" prima che arrivi a un cliente.

---
## 2. Riaddestrare v1 e v2 (come nelle lezioni 4, 7 e 8)

Come nelle lezioni precedenti, questo notebook riaddestra da zero, dentro la propria cartella effimera: non dipende da nessun artefatto salvato prima. `v1` è la configurazione senza bilanciare (quella che, nella lezione 8, era la versione "già in produzione" prima della promozione); `v2` è la configurazione vincitrice bilanciata -- il candidato che questa lezione distribuisce con cautela, invece di promuovere di colpo come ha fatto direttamente la lezione 8 riassegnando l'alias.

In [ ]:
X = df[COLUMNAS_ENTRADA]
y = df["abandono"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y
)

preprocesador = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
])

# v1: la configurazione SENZA bilanciare -- quella che, nella lezione 8, "era già in produzione" prima della promozione
clasificador_v1 = LogisticRegression(random_state=SEMILLA, max_iter=1000)
pipeline_v1 = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador_v1)])
pipeline_v1.fit(X_train, y_train)
f1_v1 = round(f1_score(y_test, pipeline_v1.predict(X_test)), 4)
auc_v1 = round(roc_auc_score(y_test, pipeline_v1.predict_proba(X_test)[:, 1]), 4)
acc_v1 = round(accuracy_score(y_test, pipeline_v1.predict(X_test)), 4)

# v2: la configurazione vincitrice (bilanciata) delle lezioni 4, 5, 7 e 8 -- il "candidato" che questa lezione distribuisce con cautela
clasificador_v2 = LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000, C=1.0)
pipeline_v2 = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador_v2)])
pipeline_v2.fit(X_train, y_train)
f1_v2 = round(f1_score(y_test, pipeline_v2.predict(X_test)), 4)
auc_v2 = round(roc_auc_score(y_test, pipeline_v2.predict_proba(X_test)[:, 1]), 4)
acc_v2 = round(accuracy_score(y_test, pipeline_v2.predict(X_test)), 4)

MODELOS = {"v1": pipeline_v1, "v2": pipeline_v2}

print("v1 (senza bilanciare, in produzione fino a oggi) -- F1:", f1_v1, "AUC:", auc_v1, "accuratezza:", acc_v1)
print("v2 (bilanciata, candidata)        -- F1:", f1_v2, "AUC:", auc_v2, "accuratezza:", acc_v2)
assert f1_v2 > f1_v1 and auc_v1 > 0.6, "il ri-addestramento non ha riprodotto i modelli attesi"

---
## 3. Costruire l'API con due versioni attive

Sulla stessa `Cliente` e sullo stesso schema della lezione 7, questo notebook espone DUE endpoint -- `/predecir_v1` e `/predecir_v2` -- che condividono lo stesso processo e lo stesso `TestClient`, simulando due versioni distribuite nello stesso momento dietro un router di traffico.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from fastapi import FastAPI
from fastapi.testclient import TestClient


class Cliente(BaseModel):
    """Lo stesso modello Pydantic della lezione 7: esattamente le colonne di input di churn.csv."""
    id_cliente: str
    antiguedad_meses: int = Field(ge=0)
    plan: Literal["basico", "estandar", "premium"]
    contrato: Literal["mensual", "anual"]
    uso_mensual_horas: float = Field(ge=0)
    tickets_soporte: int = Field(ge=0)
    metodo_pago: Literal["domiciliacion", "tarjeta", "transferencia"]
    factura_mensual: float = Field(ge=0)


def cliente_a_fila(cliente):
    return pd.DataFrame([cliente.model_dump()])[COLUMNAS_ENTRADA]


def respuesta_de(nombre_version, cliente):
    fila = cliente_a_fila(cliente)
    probabilidad = float(MODELOS[nombre_version].predict_proba(fila)[0, 1])
    return {
        "id_cliente": cliente.id_cliente,
        "version": nombre_version,
        "probabilidad_abandono": round(probabilidad, 4),
        "abandono_previsto": int(probabilidad >= 0.5),
    }


app = FastAPI(title="API di churn (lezione 7), con due versioni attive")


@app.post("/predecir_v1")
def predecir_v1(cliente: Cliente):
    return respuesta_de("v1", cliente)


@app.post("/predecir_v2")
def predecir_v2(cliente: Cliente):
    return respuesta_de("v2", cliente)


# TestClient, come nella lezione 7: senza porte né processi reali, deterministico
cliente_api = TestClient(app)

cliente_prueba = X_test.iloc[0].to_dict()
cliente_prueba["id_cliente"] = "C_TEST_0001"
r1 = cliente_api.post("/predecir_v1", json=cliente_prueba)
r2 = cliente_api.post("/predecir_v2", json=cliente_prueba)
print("/predecir_v1 ->", r1.status_code, r1.json())
print("/predecir_v2 ->", r2.status_code, r2.json())
assert r1.status_code == 200 and r2.status_code == 200
assert r1.json()["version"] == "v1" and r2.json()["version"] == "v2"
print("Le due versioni dell'API rispondono, nello stesso processo, pronte a ripartire il traffico tra loro.")

---
## 4. Un generatore di traffico sintetico riproducibile

Per confrontare le strategie servono richieste -- tante, e sempre le STESSE tra un'esecuzione e l'altra. Il notebook campiona, senza reinserimento e con un seme fisso (`SEMILLA_TRAFICO=7`, diverso dal seme del modello), 400 righe di `churn.csv` e le converte in richieste valide per `Cliente` (senza la colonna `abandono`, che è l'etichetta da prevedere, non un dato di input).

In [ ]:
N_TRAFICO = 400
# Seme fisso e distinto da SEMILLA (quello del modello): così il generatore di traffico è riproducibile di per sé
SEMILLA_TRAFICO = 7
rng_trafico = np.random.default_rng(SEMILLA_TRAFICO)
indices_trafico = rng_trafico.choice(df.index.values, size=N_TRAFICO, replace=False)
trafico_df = df.loc[indices_trafico].reset_index(drop=True)
peticiones = trafico_df[["id_cliente"] + COLUMNAS_ENTRADA].to_dict(orient="records")

print("Traffico sintetico generato:", len(peticiones), "richieste, campionate senza reinserimento da churn.csv con seme fisso.")
print("prima richiesta:", peticiones[0])

---
## 5. Strategia 1 — Canary: una piccola frazione alla versione nuova

In un rilascio **canary**, la versione candidata riceve solo una piccola frazione del traffico (qui, l'8 %); il resto resta sulla versione stabile. Se qualcosa va storto, il danno resta limitato a quella frazione. La decisione su quale versione riceve ogni richiesta si prende con un generatore casuale (`numpy.random.default_rng`) con seme fisso: casuale nel senso di "non dipende dal contenuto della richiesta", ma riproducibile tra esecuzioni.

In [ ]:
P_CANARY = 0.08  # 8 % del traffico alla versione candidata
SEMILLA_CANARY = 123
rng_canary = np.random.default_rng(SEMILLA_CANARY)


def enrutar_canario(rng):
    """Estrae, con lo STESSO seme a ogni esecuzione, se questa richiesta va a v2 (candidata) o a v1 (stabile)."""
    return "v2" if rng.random() < P_CANARY else "v1"


conteo_canary = Counter()
for peticion in peticiones:
    version = enrutar_canario(rng_canary)
    conteo_canary[version] += 1
    respuesta = cliente_api.post(f"/predecir_{version}", json=peticion)
    assert respuesta.status_code == 200

pct_v2_canary = round(100 * conteo_canary["v2"] / N_TRAFICO, 2)
print("Ripartizione canary su", N_TRAFICO, "richieste:", dict(conteo_canary))
print(f"-> {pct_v2_canary} % è andato alla versione candidata (v2), il resto alla versione stabile (v1) -- obiettivo: {P_CANARY:.0%}.")
assert conteo_canary["v1"] + conteo_canary["v2"] == N_TRAFICO

**Cosa osservi?** La ripartizione reale (ricalcolata nel notebook) resta vicina all'8 % obiettivo, ma non esattamente su di esso -- 400 estrazioni indipendenti con probabilità 0,08 non danno ESATTAMENTE 32 casi, per la stessa ragione per cui lanciare una moneta 400 volte non dà esattamente 200 teste. Con più traffico, la ripartizione osservata si avvicina di più all'obiettivo.

---
## 6. Strategia 2 — Shadow: la stessa richiesta, due risposte, un solo conteggio

In un rilascio **shadow**, la versione candidata (v2) riceve una COPIA di ogni richiesta, ma la sua risposta non arriva mai all'utente: si confronta solo con quella della versione in uso (v1). È la strategia a rischio più basso possibile -- v2 non decide mai nulla da sola -- al costo di duplicare il lavoro di calcolo (ogni richiesta viene elaborata due volte).

In [ ]:
# In shadow, TUTTE le richieste vengono inviate anche a v2, ma la risposta che "vede l'utente" è sempre quella di v1
acuerdos = 0
desacuerdos_muestra = []
for peticion in peticiones:
    respuesta_usuario = cliente_api.post("/predecir_v1", json=peticion).json()  # questo è l'unica cosa che arriva all'utente
    respuesta_sombra = cliente_api.post("/predecir_v2", json=peticion).json()   # questo si registra solo per confrontare, non si restituisce mai
    if respuesta_usuario["abandono_previsto"] == respuesta_sombra["abandono_previsto"]:
        acuerdos += 1
    elif len(desacuerdos_muestra) < 3:
        desacuerdos_muestra.append((peticion["id_cliente"], respuesta_usuario["abandono_previsto"], respuesta_sombra["abandono_previsto"]))

tasa_acuerdo = round(acuerdos / N_TRAFICO, 4)
print("Accordi v1/v2 in shadow:", acuerdos, "su", N_TRAFICO, f"-> tasso di accordo: {tasa_acuerdo:.2%}")
print("esempi di disaccordo (id_cliente, previsto v1, previsto v2):", desacuerdos_muestra)
print("L'utente NON ha mai visto la risposta di v2 in questa sezione: shadow misura senza alcun rischio per nessuno.")
assert 0.0 <= tasa_acuerdo <= 1.0

**Cosa osservi?** Il tasso di accordo ricalcolato è nettamente minore di 1: v1 (senza bilanciare) quasi non prevede mai abbandono (il suo F1 nella sezione 2 è praticamente nullo), mentre v2 prevede abbandono più spesso. Quel disaccordo NON è un errore del notebook: è la differenza reale di comportamento tra le due versioni, esattamente il tipo di segnale che shadow è pensato per esporre prima di decidere una promozione.

---
## 7. Strategia 3 — A/B: ripartizione deterministica per `id_cliente`

In un rilascio **A/B**, le due versioni sono attive e servono traffico reale nello stesso momento (a differenza di canary, dove una è chiaramente la "minoritaria"). La domanda a cui risponde non è "v2 fallisce?" ma "v2 è MIGLIORE per il business?", misurato con una metrica di business -- qui, un tasso di conversione fittizio (per esempio, accettare un'offerta di retention), non una metrica del modello come F1.

La ripartizione tra gruppi usa un **hash deterministico** dell'`id_cliente` (MD5, in questo notebook), NON un'estrazione casuale pura a ogni richiesta. La ragione è di prodotto, non tecnica: se lo stesso cliente potesse cadere in un gruppo diverso a ogni visita, vedrebbe prezzi, testi o decisioni diversi da una sessione all'altra -- un'esperienza incoerente che genererebbe reclami e contaminerebbe la misurazione stessa (la conversione è migliorata per la versione, o perché lo stesso cliente ha visto entrambe e si è confuso?). L'hash risolve questo senza dover salvare in nessun database "a quale gruppo appartiene ogni cliente": lo stesso input produce sempre lo stesso output.

In [ ]:
def grupo_ab(id_cliente):
    """Hash MD5 dell'id_cliente, non un'estrazione casuale: lo STESSO id_cliente cade SEMPRE nello stesso gruppo."""
    digest = int(hashlib.md5(id_cliente.encode()).hexdigest(), 16)
    return "v1_control" if digest % 2 == 0 else "v2_tratamiento"


# Dimostrazione: l'hash è deterministico (si ripete in due chiamate); un'estrazione con random.random() non lo sarebbe
un_id = trafico_df.loc[0, "id_cliente"]
assert grupo_ab(un_id) == grupo_ab(un_id), "l'hash dovrebbe dare sempre lo stesso gruppo per lo stesso id_cliente"
print(f"'{un_id}' cade nel gruppo", grupo_ab(un_id), "in due chiamate consecutive -- sempre lo stesso.")

### 7.1 Ripartire tutta la base clienti e simulare una metrica di business

In [ ]:
# Assegniamo il gruppo a TUTTA la base clienti (non solo al traffico della sezione 4): così, ogni cliente che torna
# domani continua a vedere la stessa versione di oggi -- la ragione di business dietro l'uso di un hash e non un numero casuale
grupos = df["id_cliente"].astype(str).map(grupo_ab)
n_control = int((grupos == "v1_control").sum())
n_tratamiento = int((grupos == "v2_tratamiento").sum())
print("dimensione del gruppo v1_control:", n_control, "| dimensione del gruppo v2_tratamiento:", n_tratamiento)
assert n_control + n_tratamiento == len(df)

# Metrica di business simulata: tasso di conversione (per esempio, accettare un'offerta di retention), NON una metrica del modello
# con probabilità diverse per gruppo (dati fittizi, creati per questo esercizio) e seme fisso
SEMILLA_CONVERSION = 42
rng_conversion = np.random.default_rng(SEMILLA_CONVERSION)
PROB_CONVERSION_CONTROL = 0.20
PROB_CONVERSION_TRATAMIENTO = 0.245
conversiones_control = rng_conversion.binomial(1, PROB_CONVERSION_CONTROL, n_control)
conversiones_tratamiento = rng_conversion.binomial(1, PROB_CONVERSION_TRATAMIENTO, n_tratamiento)

tasa_control = conversiones_control.mean()
tasa_tratamiento = conversiones_tratamiento.mean()
print(f"tasso di conversione v1_control: {tasa_control:.4f} ({int(conversiones_control.sum())}/{n_control})")
print(f"tasso di conversione v2_tratamiento: {tasa_tratamiento:.4f} ({int(conversiones_tratamiento.sum())}/{n_tratamiento})")

### 7.2 Confrontare i due gruppi con un test statistico reale

In [ ]:
# Tabella di contingenza 2x2 (converte/non converte x controllo/trattamento) e chi-quadro di indipendenza
tabla_contingencia = [
    [int(conversiones_control.sum()), n_control - int(conversiones_control.sum())],
    [int(conversiones_tratamiento.sum()), n_tratamiento - int(conversiones_tratamiento.sum())],
]
chi2, valor_p, grados_libertad, _ = chi2_contingency(tabla_contingencia, correction=False)
print("tabella di contingenza [converte, non converte] x [controllo, trattamento]:", tabla_contingencia)
print("chi-quadro:", round(chi2, 4), "valore p:", round(valor_p, 4))

# Inoltre, la differenza di proporzioni con un intervallo di confidenza del 95 % (approssimazione normale di Wald)
diferencia = tasa_tratamiento - tasa_control
error_estandar_diferencia = np.sqrt(
    tasa_control * (1 - tasa_control) / n_control + tasa_tratamiento * (1 - tasa_tratamiento) / n_tratamiento
)
ic_bajo = diferencia - 1.96 * error_estandar_diferencia
ic_alto = diferencia + 1.96 * error_estandar_diferencia
print(f"differenza (trattamento - controllo): {diferencia:.4f}, IC 95 %: [{ic_bajo:.4f}, {ic_alto:.4f}]")

ALFA = 0.05
es_significativo = valor_p < ALFA
print(f"differenza statisticamente significativa (p < {ALFA})?:", es_significativo)
assert (ic_bajo > 0) == es_significativo, "l'IC e il valore p dovrebbero coincidere nella loro conclusione"
print("L'intervallo di confidenza NON include lo 0 e il valore p è minore di 0,05: in questo esempio illustrativo, il gruppo con v2 converte di più, e non sembra casualità.")

> **Attenzione:** in questo esempio illustrativo, i tassi di conversione sono dati SIMULATI (con `rng_conversion.binomial`, seme fisso) -- non provengono da `churn.csv` né da alcun dato reale del corso, perché il dataset non include nessuna metrica di conversione. Ciò che è reale e verificabile qui è il CALCOLO del test statistico su quei dati simulati, non la cifra di business in sé.

---
## 8. Confrontare le tre strategie

| | Canary | Shadow | A/B |
|---|---|---|---|
| Chi vede la risposta di v2? | una piccola frazione di utenti reali | nessuno (solo un log di confronto) | circa metà degli utenti reali |
| Rischio se v2 ha un errore | limitato a quella frazione | nessuno | limitato al gruppo trattamento |
| Cosa misura | v2 fallisce o si comporta male in produzione reale? | in cosa si differenziano le previsioni di v1 e v2? | v2 è MIGLIORE per una metrica di business? |
| Costo di calcolo | normale (ogni richiesta va a una sola versione) | doppio (ogni richiesta viene elaborata due volte) | normale |
| Ripartizione | casuale con seme fisso | nessuna -- tutto il traffico viene duplicato | deterministica (hash di `id_cliente`) |

**Regola pratica:** prima shadow (rischio zero, per sapere se v2 si comporta in modo ragionevolmente diverso), poi canary (rischio limitato, per confermare che non fallisce con traffico reale), infine A/B (quando ci si fida già che v2 non fallisca, e resta da decidere solo se è meglio per il business).

---
## 9. Rollback: tornare indietro quando una metrica peggiora

Un **rollback** torna alla versione precedente quando quella nuova causa un problema. Perché non sia una decisione improvvisata, serve un **criterio esplicito**: una metrica concreta, una versione di riferimento e una soglia. Qui: se l'F1 misurato su un lotto recente scende di oltre il 15 % (relativo) rispetto all'F1 già validato, si torna alla versione precedente.

In [ ]:
UMBRAL_CAIDA_RELATIVA = 0.15  # 15 %: criterio di decisione esplicito di questo notebook


def decidir_rollback(metrica_base, metrica_actual, umbral_relativo=UMBRAL_CAIDA_RELATIVA):
    """Restituisce (debe_revertir, caida_relativa, motivo). "metrica_base" è la metrica già validata (per esempio F1 sul set di test); "metrica_actual" è la stessa metrica misurata su traffico più recente."""
    caida_relativa = (metrica_base - metrica_actual) / metrica_base
    debe_revertir = caida_relativa > umbral_relativo
    if debe_revertir:
        motivo = f"caduta relativa di {caida_relativa:.2%} SUPERA la soglia di {umbral_relativo:.0%}"
    else:
        motivo = f"caduta relativa di {caida_relativa:.2%} NON supera la soglia di {umbral_relativo:.0%}"
    return debe_revertir, round(caida_relativa, 4), motivo


def registrar_decision(version_actual, version_anterior, debe_revertir, caida_relativa, motivo):
    """Un log reale della decisione (senza orologio di sistema, perché sia riproducibile): cosa si è deciso, su quale versione e perché."""
    entrada = {
        "accion": "rollback" if debe_revertir else "mantener_version_actual",
        "version_en_uso_antes_de_decidir": version_actual,
        "version_restaurada": version_anterior if debe_revertir else version_actual,
        "caida_relativa": caida_relativa,
        "umbral": UMBRAL_CAIDA_RELATIVA,
        "motivo": motivo,
    }
    print(json.dumps(entrada, ensure_ascii=False))
    return entrada

### 9.1 Caso normale: una variazione piccola NON fa scattare il rollback

In [ ]:
# Caso 1: un lotto di traffico recente con una variazione piccola e normale (15 % delle righe con un dato di utilizzo mancante, per esempio un sensore che a volte fallisce)
rng_variacion = np.random.default_rng(SEMILLA)
indices_afectados_leve = rng_variacion.choice(X_test.index, size=int(len(X_test) * 0.15), replace=False)
X_test_variacion_leve = X_test.copy()
X_test_variacion_leve.loc[indices_afectados_leve, "uso_mensual_horas"] = 0.0
f1_v2_variacion_leve = round(f1_score(y_test, pipeline_v2.predict(X_test_variacion_leve)), 4)
print("F1 di v2 sul lotto con variazione lieve:", f1_v2_variacion_leve, "(base:", f1_v2, ")")

debe_revertir_1, caida_1, motivo_1 = decidir_rollback(f1_v2, f1_v2_variacion_leve)
registrar_decision("v2", "v1", debe_revertir_1, caida_1, motivo_1)
assert debe_revertir_1 is False, "una variazione così piccola non dovrebbe far scattare un rollback"

### 9.2 Caso degradato: un errore reale di dati fa scattare il rollback

In [ ]:
# Caso 2: un errore reale della pipeline di feature -- "uso_mensual_horas" arriva sempre a 0 per TUTTO il lotto (per esempio il servizio che la calcola si è fermato)
X_test_degradado = X_test.copy()
X_test_degradado["uso_mensual_horas"] = 0.0
f1_v2_degradado = round(f1_score(y_test, pipeline_v2.predict(X_test_degradado)), 4)
print("F1 di v2 sul lotto degradato:", f1_v2_degradado, "(base:", f1_v2, ")")

debe_revertir_2, caida_2, motivo_2 = decidir_rollback(f1_v2, f1_v2_degradado)
entrada_log = registrar_decision("v2", "v1", debe_revertir_2, caida_2, motivo_2)
assert debe_revertir_2 is True, "questa caduta dovrebbe far scattare il rollback"
assert entrada_log["accion"] == "rollback" and entrada_log["version_restaurada"] == "v1"
print("Rollback eseguito: il traffico torna a v1 finché la pipeline di feature non viene corretta e il rilascio graduale non viene ripetuto dalla sezione 5.")

**Cosa osservi?** La STESSA funzione `decidir_rollback` con la STESSA soglia (15 %) dà due decisioni diverse perché la caduta reale è diversa: ~4 % nel caso normale (non fa scattare nulla), ~21 % nel caso degradato (fa scattare il rollback). Il criterio non cambia; cambia il dato che entra. Il log (una riga JSON) resta come evidenza di cosa si è deciso e perché, senza dipendere da nessun orologio di sistema.

---
## 10. Errori comuni

1. **Ripartire il traffico A/B con un'estrazione casuale a ogni richiesta, invece di un hash dell'`id_cliente`**: lo stesso cliente vedrebbe una versione diversa a ogni visita, rompendo l'esperienza e contaminando la metrica di business stessa (sezione 7).
2. **Confondere shadow con canary**: in canary, v2 decide DAVVERO (per una frazione di utenti); in shadow, v2 non decide MAI -- si confronta soltanto. Trattarli come sinonimi porta a sottostimare il rischio reale di un rilascio canary.
3. **Promuovere un candidato a produzione solo perché "non ha fallito" in canary**, senza guardare una metrica di business (A/B) né confrontare con la versione precedente con un test statistico: "non ha fallito" non è lo stesso di "è migliore".
4. **Fissare la soglia di rollback guardando i dati DOPO aver visto il problema** ("aggiustiamo la soglia perché oggi non scatti"): la soglia va decisa PRIMA, con la metrica base già validata (sezione 9), altrimenti smette di essere un criterio e diventa una scusa.
5. **Ignorare la dimensione di ogni gruppo in un test A/B**: con campioni molto piccoli, anche una differenza reale può non essere statisticamente significativa (intervallo di confidenza ampio, che include lo 0) -- la sezione 7.2 lo verifica esplicitamente con l'intervallo di confidenza, non solo con il valore p.

---
## 11. Checklist prima di considerare pronto un rilascio

- [ ] So a cosa serve ogni ambiente (sviluppo, staging, produzione) e non confondo il provare in staging con l'essere in produzione.
- [ ] Il candidato è passato prima per shadow (rischio zero) prima di ricevere traffico reale.
- [ ] Se usa canary, la frazione di traffico è fissata in anticipo e la ripartizione reale si misura, non si assume.
- [ ] Se usa A/B, la ripartizione è deterministica per `id_cliente` (o un altro identificativo cliente stabile), non un'estrazione per richiesta.
- [ ] Il confronto A/B usa un test statistico reale (valore p, intervallo di confidenza), non solo "il numero più alto ha vinto".
- [ ] C'è un criterio di rollback esplicito (metrica, versione base, soglia) deciso PRIMA di vedere il problema.
- [ ] Il rollback lascia un log reale della decisione (cosa, perché, con quali cifre), senza dipendere da un orologio di sistema.
- [ ] Nessuna cartella di lavoro effimera (`proyecto_churn/`) è rimasta dentro il repository del corso: verifico con `git status`.

---
## 12. Domande di autovalutazione

1. Qual è la differenza essenziale tra canary e shadow riguardo a CHI vede la risposta della versione candidata?
2. **Calcolo.** Con `UMBRAL_CAIDA_RELATIVA=0.15` e un F1 base di 0,3166, a partire da quale F1 (arrotondato a 4 decimali) dovrebbe scattare il rollback? *(Suggerimento: risolvi `(0,3166 - x) / 0,3166 > 0,15` per il valore limite di `x`.)*
3. Perché la ripartizione A/B usa un hash dell'`id_cliente` e non un numero casuale diverso a ogni richiesta? Spiega la conseguenza concreta per l'esperienza di uno stesso cliente.
4. Nella sezione 7.2, cosa significa che l'intervallo di confidenza della differenza di proporzioni NON includa lo 0? È lo stesso che il valore p sia minore di 0,05?
5. Un collega propone di promuovere un candidato a produzione appena finisce un canary di un giorno senza errori del server (nessun codice 500). Cosa gli mancherebbe da verificare, usando il ragionamento delle sezioni 6, 7 e 9?

*(Risposta orientativa alla domanda 2: `(0,3166 - x) / 0,3166 = 0,15` implica `x = 0,3166 * (1 - 0,15) = 0,26911`; con F1 sotto circa 0,2691 il rollback dovrebbe scattare.)*

---
## Pulizia finale

In [ ]:
shutil.rmtree(CARPETA)
print("Cartella di lavoro eliminata:", CARPETA)
assert not os.path.exists(CARPETA)